In [38]:
import re
import random
from collections import Counter, defaultdict

import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
nltk.download('punkt_tab')

df = pd.read_csv('leesmateriaal_Opgeschoond2.csv')
df['tekst_nlp'] = df['tekst_nlp'].fillna('')

stopwoorden_nl = set(stopwords.words('dutch'))
stemmer = SnowballStemmer('dutch')

def opschonen(tekst):
    tekst = re.sub(r'[^a-zà-ÿ\s]', ' ', str(tekst).lower())
    return re.sub(r'\s+', ' ', tekst).strip()

def tokens_schoon(tekst):
    return [w for w in word_tokenize(opschonen(tekst), language='dutch')
            if w not in stopwoorden_nl and len(w) > 1]

def nlp_pipeline(tekst):
    return ' '.join(stemmer.stem(w) for w in tokens_schoon(tekst))

print(df.shape)

(283, 31)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aswei\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aswei\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [39]:
tellers = defaultdict(Counter)
for tekst in df['gecombineerde_tekst']:
    for w in tokens_schoon(tekst):
        tellers[stemmer.stem(w)][w] += 1

stam_naar_woord = {stam: teller.most_common(1)[0][0] for stam, teller in tellers.items()}

for stam in ['avontur', 'vriendschap', 'oorlog', 'spannend']:
    print(f'{stam:12} -> {stam_naar_woord.get(stam)}')

avontur      -> avontuur
vriendschap  -> vriendschap
oorlog       -> oorlog
spannend     -> spannende


In [40]:
vectorizer = TfidfVectorizer()
X_boeken = vectorizer.fit_transform(df['tekst_nlp'])
termen = vectorizer.get_feature_names_out()

NIVEAU_WAARDE = {'2F': 2.0, '2F-3F': 2.5, '3F': 3.0, '3F+': 3.5}

def niveau_past(boek_niveau, student_niveau):
    b, s = NIVEAU_WAARDE.get(boek_niveau), NIVEAU_WAARDE.get(student_niveau)
    return b is not None and s is not None and abs(b - s) <= 0.5

def aanbevelen(profiel, k=3):
    v = vectorizer.transform([nlp_pipeline(profiel['interesses'])])
    scores = cosine_similarity(v, X_boeken).ravel()
    kandidaten = df.assign(score=scores)
    kandidaten = kandidaten[kandidaten['niveau'].apply(lambda n: niveau_past(n, profiel['niveau']))]
    if profiel.get('genre'):
        # voorkeursgenre krijgt een kleine bonus, maar andere genres blijven mogelijk (variatie)
        kandidaten['score'] += 0.05 * (kandidaten['genre'] == profiel['genre'])
    kandidaten = kandidaten.drop_duplicates('titel')
    return kandidaten.sort_values('score', ascending=False).head(k), v

In [41]:
def redenen(boek_idx, profiel_vec, profiel):
    boek_vec = X_boeken[boek_idx].toarray().ravel()
    profiel_arr = profiel_vec.toarray().ravel()
    gedeeld = [(termen[i], boek_vec[i]) for i in range(len(termen)) if boek_vec[i] > 0 and profiel_arr[i] > 0]
    gedeeld = [stam_naar_woord.get(t, t) for t, _ in sorted(gedeeld, key=lambda x: -x[1])][:3]

    boek = df.loc[boek_idx]
    verschil = NIVEAU_WAARDE[boek['niveau']] - NIVEAU_WAARDE[profiel['niveau']]
    return {
        'titel': boek['titel'],
        'auteur': boek['auteur'],
        'type': str(boek['type_materiaal']).lower(),
        'gedeelde_termen': gedeeld,
        'genre_match': profiel.get('genre') == boek['genre'],
        'genre': boek['genre'],
        'niveau': boek['niveau'],
        'niveau_verschil': verschil,
        'leestijd': boek['geschatte_leestijd_min'],
    }

In [42]:
def opsomming(woorden):
    woorden = [f'**{w}**' for w in woorden]
    return woorden[0] if len(woorden) == 1 else ', '.join(woorden[:-1]) + ' en ' + woorden[-1]

def leestijd_tekst(minuten):
    if pd.isna(minuten):
        return None
    minuten = int(minuten)
    if minuten < 60:
        return f'Je leest het in ongeveer {minuten} minuten.'
    uren = round(minuten / 60)
    return f'Je bent er ongeveer {uren} uur mee bezig.'

def genereer_uitleg(r, rnd):
    zinnen = []

    # 1. inhoud / interesses
    if r['gedeelde_termen']:
        zinnen.append(rnd.choice([
            f"Dit past bij jou omdat het gaat over {opsomming(r['gedeelde_termen'])}.",
            f"Je gaf aan dat je {opsomming(r['gedeelde_termen'])} interessant vindt. Daar gaat dit over.",
            f"In dit {r['type']} {'komt' if len(r['gedeelde_termen']) == 1 else 'komen'} {opsomming(r['gedeelde_termen'])} aan bod, net als in jouw interesses.",
        ]))
    else:
        zinnen.append("Dit sluit niet direct aan op je interesses, maar het is een kans om iets nieuws te ontdekken.")

    # 2. genre
    if r['genre_match']:
        zinnen.append(f"Het is {r['genre']}, het genre dat je het liefst leest.")
    else:
        zinnen.append(f"Het is {r['genre']}: eens iets anders dan je gewend bent.")

    # 3. niveau
    if r['niveau_verschil'] == 0:
        zinnen.append(f"Het niveau ({r['niveau']}) past precies bij jou.")
    elif r['niveau_verschil'] > 0:
        zinnen.append(f"Het niveau ({r['niveau']}) is iets hoger dan jouw niveau: een mooie uitdaging.")
    else:
        zinnen.append(f"Het niveau ({r['niveau']}) is iets makkelijker, dus je leest het lekker vlot.")

    # 4. leestijd
    lt = leestijd_tekst(r['leestijd'])
    if lt:
        zinnen.append(lt)

    return ' '.join(zinnen)

def leesadvies(profiel, k=3, seed=1):
    rnd = random.Random(seed)   # vaste seed: zelfde profiel geeft zelfde uitleg (reproduceerbaar)
    top, v = aanbevelen(profiel, k)
    print(f"Leesadvies voor {profiel['naam']}  (interesses: {profiel['interesses']}, niveau {profiel['niveau']})")
    print('-' * 80)
    for nr, (idx, rij) in enumerate(top.iterrows(), 1):
        r = redenen(idx, v, profiel)
        print(f"{nr}. {r['titel']} – {r['auteur']}  (match {rij['score']:.0%})")
        print('   ' + genereer_uitleg(r, rnd))
        print()

In [43]:
profielen = [
    {'naam': 'Sara',  'interesses': 'oorlog geschiedenis verlies familie', 'niveau': '3F', 'genre': 'fictie'},
    {'naam': 'Mehmet','interesses': 'voetbal sport vriendschap spannend',   'niveau': '2F', 'genre': 'fictie'},
    {'naam': 'Lina',  'interesses': 'gezondheid werk geld toekomst',        'niveau': '2F', 'genre': 'informatief'},
]
for p in profielen:
    leesadvies(p)


Leesadvies voor Sara  (interesses: oorlog geschiedenis verlies familie, niveau 3F)
--------------------------------------------------------------------------------
1. De val – Marga Minco  (match 54%)
   Dit past bij jou omdat het gaat over **oorlog** en **verlies**. Het is fictie, het genre dat je het liefst leest. Het niveau (3F) past precies bij jou. Je bent er ongeveer 9 uur mee bezig.

2. Ik zal je vinden – Monica Hesse  (match 47%)
   In dit boek komen **familie** en **oorlog** aan bod, net als in jouw interesses. Het is fictie, het genre dat je het liefst leest. Het niveau (3F) past precies bij jou. Je bent er ongeveer 8 uur mee bezig.

3. Moeder mijn vrouw – Yvonne Keuls  (match 31%)
   Dit past bij jou omdat het gaat over **familie**. Het is fictie, het genre dat je het liefst leest. Het niveau (3F) past precies bij jou. Je bent er ongeveer 14 uur mee bezig.

Leesadvies voor Mehmet  (interesses: voetbal sport vriendschap spannend, niveau 2F)
-----------------------------------